# Azure AI Search: Full-Text Search and RAG

In this notebook you will learn how to use **Azure AI Search** to:

1. Create a search index and upload sample documents
2. Query an index with **full-text search**
3. Apply **filters** to narrow results
4. Combine Azure AI Search with Azure OpenAI for **Retrieval Augmented Generation (RAG)**

---

### Prerequisites: Deploy the Azure Resources

This notebook requires **two** Azure resources: **Azure AI Search** and **Azure OpenAI**. Both are provisioned by a single ARM template included in this folder.

> **Tip:** Open `deploy-search.parameters.json` to customize the resource names, region, search SKU, or OpenAI model before deploying.

**1. Deploy the resources** — run the cell below. Set `RESOURCE_GROUP` in `.env` if your resource group is not named `Foundry`.

This single deployment creates:
1. **Azure AI Search** service (with semantic search and Entra ID auth enabled)
2. **Azure OpenAI** resource with a model deployment (default: `gpt-4.1`)

In [ ]:
import os
import subprocess
import shutil
from pathlib import Path


def read_env_value(name: str, default: str | None = None) -> str | None:
    value = os.getenv(name)
    if value:
        return value

    env_path = Path(".env")
    if env_path.exists():
        for raw_line in env_path.read_text(encoding="utf-8").splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, raw_value = line.split("=", 1)
            if key.strip() == name:
                return raw_value.strip().strip('"').strip("'")

    return default


RESOURCE_GROUP = read_env_value("RESOURCE_GROUP", "Foundry")

# Resolve the full path to the Azure CLI (needed for Windows where az is a .cmd file)
AZ = shutil.which("az")
if not AZ:
    raise FileNotFoundError("Azure CLI (az) not found. Install it from https://learn.microsoft.com/cli/azure/install-azure-cli")

result = subprocess.run(
    [AZ, "deployment", "group", "create",
     "--resource-group", RESOURCE_GROUP,
     "--template-file", "deploy-search.json",
     "--parameters", "@deploy-search.parameters.json"],
    capture_output=True, text=True, encoding="utf-8"
)
print(result.stdout)
if result.returncode != 0:
    print("ERROR:", result.stderr)

**2. Assign the required RBAC roles** (4 roles across 2 resources).

Run the cell below to assign all roles at once. It retrieves your user ID and the resource IDs from the deployment output, then creates the role assignments. Set `RESOURCE_GROUP` in `.env` if your resource group is not named `Foundry`.

> **Note:** RBAC role assignments can take **1–2 minutes** to propagate. If you get a Forbidden error on your first run, wait a moment and try again.

In [ ]:
import os
import subprocess
import shutil
from pathlib import Path


def read_env_value(name: str, default: str | None = None) -> str | None:
    value = os.getenv(name)
    if value:
        return value

    env_path = Path(".env")
    if env_path.exists():
        for raw_line in env_path.read_text(encoding="utf-8").splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, raw_value = line.split("=", 1)
            if key.strip() == name:
                return raw_value.strip().strip('"').strip("'")

    return default


RESOURCE_GROUP = read_env_value("RESOURCE_GROUP", "Foundry")

AZ = shutil.which("az")
if not AZ:
    raise FileNotFoundError("Azure CLI (az) not found. Install it from https://learn.microsoft.com/cli/azure/install-azure-cli")

# Step A: Get the signed-in user's Object ID
user_result = subprocess.run(
    [AZ, "ad", "signed-in-user", "show", "--query", "id", "-o", "tsv"],
    capture_output=True, text=True, encoding="utf-8"
)
user_id = user_result.stdout.strip()
print(f"Your User ID: {user_id}")

# Step B: Get the Search resource ID from the deployment output
search_result = subprocess.run(
    [AZ, "deployment", "group", "show",
     "--resource-group", RESOURCE_GROUP,
     "--name", "deploy-search",
     "--query", "properties.outputs.searchServiceId.value", "-o", "tsv"],
    capture_output=True, text=True, encoding="utf-8"
)
search_id = search_result.stdout.strip()
print(f"Search ID:    {search_id}")

# Step C: Get the OpenAI resource ID from the deployment output
openai_result = subprocess.run(
    [AZ, "deployment", "group", "show",
     "--resource-group", RESOURCE_GROUP,
     "--name", "deploy-search",
     "--query", "properties.outputs.openAIResourceId.value", "-o", "tsv"],
    capture_output=True, text=True, encoding="utf-8"
)
openai_id = openai_result.stdout.strip()
print(f"OpenAI ID:    {openai_id}")

# Step D: Assign the 4 RBAC roles
roles = [
    ("Search Service Contributor",     search_id),
    ("Search Index Data Contributor",  search_id),
    ("Search Index Data Reader",       search_id),
    ("Cognitive Services OpenAI User", openai_id),
]

for role_name, scope in roles:
    r = subprocess.run(
        [AZ, "role", "assignment", "create",
         "--assignee", user_id,
         "--role", role_name,
         "--scope", scope],
        capture_output=True, text=True, encoding="utf-8"
    )
    status = "OK" if r.returncode == 0 else "FAILED"
    print(f"  [{status}] {role_name}")
    if r.returncode != 0:
        print(f"        {r.stderr.strip()}")

**3. Update the `.env` file** in this folder with the values from the deployment output:

```
SEARCH_ENDPOINT="https://<your-search-name>.search.windows.net"
SEARCH_INDEX_NAME=""
AZURE_OPENAI_ENDPOINT="https://<your-openai-name>.openai.azure.com/"
AZURE_OPENAI_DEPLOYMENT="gpt-4-1"
```

> **Note:** Leave `SEARCH_INDEX_NAME` empty for now — Step 3 will create the index and automatically update the `.env` file for you.

You can retrieve the endpoint values by running the cell below. Set `RESOURCE_GROUP` in `.env` if your resource group is not named `Foundry`.

In [ ]:
import os
import subprocess
import shutil
from pathlib import Path


def read_env_value(name: str, default: str | None = None) -> str | None:
    value = os.getenv(name)
    if value:
        return value

    env_path = Path(".env")
    if env_path.exists():
        for raw_line in env_path.read_text(encoding="utf-8").splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, raw_value = line.split("=", 1)
            if key.strip() == name:
                return raw_value.strip().strip('"').strip("'")

    return default


RESOURCE_GROUP = read_env_value("RESOURCE_GROUP", "Foundry")

AZ = shutil.which("az")
if not AZ:
    raise FileNotFoundError("Azure CLI (az) not found. Install it from https://learn.microsoft.com/cli/azure/install-azure-cli")

result = subprocess.run(
    [AZ, "deployment", "group", "show",
     "--resource-group", RESOURCE_GROUP,
     "--name", "deploy-search",
     "--query", "properties.outputs"],
    capture_output=True, text=True, encoding="utf-8"
)
print(result.stdout)
if result.returncode != 0:
    print("ERROR:", result.stderr)

### Authentication

This notebook uses **Microsoft Entra ID** (`DefaultAzureCredential`) instead of API keys.
Make sure you are logged in with `az login`.

## Step 1: Install the Required SDKs

In [ ]:
%pip install azure-search-documents==11.7.0b2 openai azure-identity python-dotenv

## Step 2: Load Environment Variables

In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential

load_dotenv(override=True)

# Azure AI Search settings
search_endpoint = os.getenv("SEARCH_ENDPOINT")
search_index = os.getenv("SEARCH_INDEX_NAME")

# Azure OpenAI settings (for RAG)
aoai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
aoai_deployment = os.getenv("AZURE_OPENAI_DEPLOYMENT")

# Single credential for all services
credential = DefaultAzureCredential()

print("Search Endpoint:", search_endpoint)
print("Index Name:", search_index)
print("OpenAI Endpoint:", aoai_endpoint)

## Step 3: Create a Search Index and Upload Sample Documents

Before we can search, we need an index with data. Below we create a
`demo-articles` index with `title`, `content`, and `category` fields,
then upload five sample articles.

> **Note:** This cell automatically updates `SEARCH_INDEX_NAME` in your `.env` file after creating the index.

In [ ]:
import time
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    SearchIndex, SimpleField, SearchableField, SearchFieldDataType,
    SemanticConfiguration, SemanticSearch, SemanticField, SemanticPrioritizedFields,
)
from azure.search.documents import SearchClient as UploadClient

# Define the index schema
index_name = "demo-articles"
index_client = SearchIndexClient(endpoint=search_endpoint, credential=credential)

fields = [
    SimpleField(name="id", type=SearchFieldDataType.String, key=True, filterable=True),
    SearchableField(name="title", type=SearchFieldDataType.String, filterable=True, sortable=True),
    SearchableField(name="content", type=SearchFieldDataType.String),
    SimpleField(name="category", type=SearchFieldDataType.String, filterable=True, facetable=True),
]

semantic_config = SemanticConfiguration(
    name="my-semantic-config",
    prioritized_fields=SemanticPrioritizedFields(
        content_fields=[SemanticField(field_name="content")],
        title_field=SemanticField(field_name="title"),
    ),
)

index = SearchIndex(
    name=index_name,
    fields=fields,
    semantic_search=SemanticSearch(configurations=[semantic_config]),
)

result = index_client.create_or_update_index(index)
print(f"Index created: {result.name}")

# Upload sample documents
upload_client = UploadClient(endpoint=search_endpoint, index_name=index_name, credential=credential)

documents = [
    {
        "id": "1",
        "title": "Introduction to Machine Learning",
        "content": (
            "Machine learning is a subset of artificial intelligence that enables "
            "systems to learn and improve from experience. It uses algorithms to "
            "parse data, learn from it, and make decisions. Applications include "
            "image recognition, natural language processing, recommendation systems, "
            "and autonomous vehicles."
        ),
        "category": "Technology",
    },
    {
        "id": "2",
        "title": "Cloud Computing Fundamentals",
        "content": (
            "Cloud computing delivers computing services over the internet including "
            "servers, storage, databases, networking, software, and analytics. The key "
            "benefits are cost reduction, scalability, performance, reliability, and "
            "security. Major providers include Microsoft Azure, Amazon Web Services, "
            "and Google Cloud Platform."
        ),
        "category": "Technology",
    },
    {
        "id": "3",
        "title": "Sustainable Energy Solutions",
        "content": (
            "Sustainable energy encompasses solar, wind, hydroelectric, and geothermal "
            "power sources. These renewable energy technologies are crucial for reducing "
            "greenhouse gas emissions and combating climate change. Investment in clean "
            "energy has grown significantly in recent years."
        ),
        "category": "Science",
    },
    {
        "id": "4",
        "title": "Modern Software Development Practices",
        "content": (
            "Agile methodologies, DevOps, and continuous integration have transformed "
            "software development. Teams now deploy code multiple times a day using "
            "automated pipelines. Containerization with Docker and orchestration with "
            "Kubernetes enable reliable deployments across cloud environments."
        ),
        "category": "Technology",
    },
    {
        "id": "5",
        "title": "Data Privacy and Security",
        "content": (
            "Data privacy regulations like GDPR and CCPA require organizations to protect "
            "personal information. Security best practices include encryption, access "
            "controls, regular audits, and employee training. Zero-trust architecture is "
            "becoming the standard approach to enterprise security."
        ),
        "category": "Security",
    },
]

upload_result = upload_client.upload_documents(documents=documents)
succeeded = sum(1 for r in upload_result if r.succeeded)
print(f"Uploaded {succeeded}/{len(documents)} documents")

# Automatically update .env with the index name so Step 4 picks it up
env_path = ".env"
with open(env_path, "r", encoding="utf-8") as f:
    env_content = f.read()

if 'SEARCH_INDEX_NAME=""' in env_content:
    env_content = env_content.replace('SEARCH_INDEX_NAME=""', f'SEARCH_INDEX_NAME="{index_name}"')
elif "SEARCH_INDEX_NAME=" in env_content:
    import re
    env_content = re.sub(r'SEARCH_INDEX_NAME="[^"]*"', f'SEARCH_INDEX_NAME="{index_name}"', env_content)
else:
    env_content += f'\nSEARCH_INDEX_NAME="{index_name}"\n'

with open(env_path, "w", encoding="utf-8") as f:
    f.write(env_content)

print(f"\n.env updated: SEARCH_INDEX_NAME=\"{index_name}\"")

# Allow a few seconds for the index to process the documents
time.sleep(3)
print("Index is ready — proceed to Step 4.")


## Step 4: Create the Search Client and Run a Full-Text Search

Now that the index is populated (and `.env` has been updated automatically),
we reload the environment and create a `SearchClient` to run a full-text search.

In [ ]:
from azure.search.documents import SearchClient
from azure.search.documents.models import QueryType

# Reload .env to pick up SEARCH_INDEX_NAME written by Step 3
load_dotenv(override=True)
search_index = os.getenv("SEARCH_INDEX_NAME")

# Fallback: if the .env value is still empty, use the variable from Step 3
if not search_index:
    search_index = index_name  # defined in Step 3
    print(f"(Using index_name from Step 3: '{search_index}')")

search_client = SearchClient(
    endpoint=search_endpoint,
    index_name=search_index,
    credential=credential,
)
print(f"Search client connected to index '{search_index}'")

# Run a full-text search query
query = "machine learning applications"

results = search_client.search(
    search_text=query,
    top=5,                       # Return the top 5 results
    include_total_count=True,    # Include the total number of matches
)

print(f"\nSearch: '{query}'")
print(f"Total matches: {results.get_count()}\n")

for i, result in enumerate(results, start=1):
    score = result.get("@search.score", "N/A")
    print(f"  {i}. [Score: {score}]")
    for key, value in result.items():
        if not key.startswith("@"):
            display_val = str(value)[:120]
            print(f"     {key}: {display_val}")
    print()
print("Semantic search using 'my-semantic-config':")
semantic_results = search_client.search(
    search_text=query,
    query_type=QueryType.SEMANTIC,
    semantic_configuration_name="my-semantic-config",
    top=3,
)

for i, result in enumerate(semantic_results, start=1):
    reranker_score = result.get("@search.reranker_score", "N/A")
    print(f"  {i}. [Reranker: {reranker_score}]")
    print(f"     title: {result.get('title')}")
    print(f"     category: {result.get('category')}")
    print()


## Step 5: Filtered Search

You can combine a text query with **OData filters** to narrow results.
For example, filter by category, date range, or any indexed field.

> **Note:** The filter syntax depends on the fields in your index.
> Adjust the filter expression below to match your schema.

In [ ]:
# Filter to only Technology articles
filter_expression = "category eq 'Technology'"

filtered_results = search_client.search(
    search_text="cloud computing",
    filter=filter_expression,
    top=3,
)

print(f"Filtered search (category = 'Technology'):")
for i, result in enumerate(filtered_results, start=1):
    score = result.get("@search.score", "N/A")
    print(f"  {i}. [Score: {score}]")
    for key, value in result.items():
        if not key.startswith("@"):
            display_val = str(value)[:120]
            print(f"     {key}: {display_val}")
    print()

## Step 6: Retrieval Augmented Generation (RAG)

RAG combines **search retrieval** with **LLM generation**. We ask Azure
OpenAI a question and attach our Azure AI Search index as a data source.
The model will ground its answer in the documents from the index.

> **Note:** This step requires both Azure AI Search **and** Azure OpenAI
> to be configured in the `.env` file. We use `get_bearer_token_provider`
> so the OpenAI client authenticates via Entra ID as well.

In [ ]:
from openai import AzureOpenAI
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

# Create a token provider for the OpenAI client
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

# Create the Azure OpenAI client using Entra ID
openai_client = AzureOpenAI(
    api_version="2024-12-01-preview",
    azure_endpoint=aoai_endpoint,
    azure_ad_token_provider=token_provider,
)

# For the data source, we also need a bearer token for the search service
search_token = credential.get_token("https://search.azure.com/.default").token

# Configure the search index as a data source for RAG
rag_config = {
    "data_sources": [
        {
            "type": "azure_search",
            "parameters": {
                "endpoint": search_endpoint,
                "index_name": search_index,
                "authentication": {
                    "type": "access_token",
                    "access_token": search_token,
                },
            },
        }
    ],
}

# Ask a question -- the model will search the index for relevant
# documents and use them to formulate its answer
user_question = "What are the key benefits of cloud computing?"

rag_response = openai_client.chat.completions.create(
    messages=[
        {"role": "user", "content": user_question},
    ],
    max_tokens=4096,
    temperature=0.7,
    model=aoai_deployment,
    extra_body=rag_config,
)

print(f"Question: {user_question}\n")
print(f"RAG Answer:\n{rag_response.choices[0].message.content}")